# MedPredict s.r.l. — Previsione della Progressione del Diabete

**Caso d'uso aziendale:** Previsione della progressione del diabete in pazienti a rischio.

**Obiettivo:** Sviluppare un modello di regressione predittiva che, utilizzando dati clinici dei pazienti, fornisca previsioni accurate sulla progressione della malattia nel tempo.

**Dataset:** Diabetes dataset (scikit-learn) — 442 pazienti, 10 variabili cliniche, target quantitativo di progressione della malattia.

---

### Variabili indipendenti
| Feature | Descrizione |
|---------|-------------|
| age | Età del paziente |
| sex | Genere |
| bmi | Indice di massa corporea |
| bp | Pressione sanguigna media |
| s1 | Colesterolo sierico totale |
| s2 | Lipoproteine a bassa densità (LDL) |
| s3 | Lipoproteine ad alta densità (HDL) |
| s4 | Rapporto colesterolo totale / HDL |
| s5 | Trigliceridi (log) |
| s6 | Livello di glicemia |

**Target:** misura quantitativa della progressione del diabete a un anno dalla raccolta dati.

# Moduli

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Metodi comuni

La funzione `print_correlation_matrix` visualizza la matrice di correlazione come una heatmap. Prende in input un DataFrame `df` e utilizza Seaborn per generare una heatmap con la scala di colore `crest`. I valori degli indici di Pearson vengono mostrati all'interno delle celle.

In [ ]:
def print_correlation_matrix(df):
    """
    Print correlation matrix as a heatmap.

    Args:
        df (DataFrame): Input DataFrame.

    Returns:
        None
    """
    plt.figure(figsize=(12, 10), dpi=100)
    sns.set_style("whitegrid")
    sns.heatmap(
        df.corr(),
        cmap="crest",
        cbar=True,
        square=True,
        yticklabels=df.columns,
        xticklabels=df.columns,
        annot=True,
        annot_kws={'size': 10},
        fmt='.2f'
    )
    plt.tight_layout()
    plt.show()

La funzione `scale_features` esegue la standardizzazione delle features utilizzando `StandardScaler`. Applica il fit sui dati di training e trasforma sia training che test, evitando data leakage.

In [ ]:
def scale_features(X_train, X_test):
    """
    Scale features using StandardScaler.

    Args:
        X_train (array-like): Training data.
        X_test (array-like): Test data.

    Returns:
        Tuple: (X_train_scaled, X_test_scaled)
    """
    ss = StandardScaler()
    X_train_scaled = ss.fit_transform(X_train)
    X_test_scaled = ss.transform(X_test)
    return X_train_scaled, X_test_scaled

La funzione `plot_predictions` visualizza i valori predetti rispetto ai valori reali tramite uno scatter plot. Una retta tratteggiata rossa indica la previsione perfetta (predetto = reale). Più i punti sono allineati sulla retta, migliore è il modello.

In [ ]:
def plot_predictions(y_test, y_pred, title=""):
    """
    Plot predicted vs actual values.

    Args:
        y_test (array-like): True values.
        y_pred (array-like): Predicted values.
        title (str): Plot title.

    Returns:
        None
    """
    plt.figure(figsize=(6, 5), dpi=100)
    plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.4)
    lim = [min(y_test.min(), y_pred.min()) - 5, max(y_test.max(), y_pred.max()) + 5]
    plt.plot(lim, lim, 'r--', lw=2, label='Previsione perfetta')
    plt.xlabel('Valori Reali')
    plt.ylabel('Valori Predetti')
    plt.title(f'Predetto vs Reale — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

La funzione `evaluate_model` valuta le prestazioni del modello di regressione. Calcola e stampa MSE, RMSE e R² sia per i dati di training che di test, e visualizza il grafico predetto vs reale per il test set.

In [ ]:
def evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test):
    """
    Evaluate regression model performance.

    Args:
        model_name (str): Name of the model.
        y_train, y_test (array-like): True values.
        y_pred_train, y_pred_test (array-like): Predicted values.

    Returns:
        None
    """
    for split, y_true, y_pred in [("TRAIN", y_train, y_pred_train), ("TEST", y_test, y_pred_test)]:
        mse  = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_true, y_pred)
        print(f"{model_name} | {split}")
        print(f"  MSE  : {mse:.2f}")
        print(f"  RMSE : {rmse:.2f}")
        print(f"  R²   : {r2:.4f}")
        print()

    plot_predictions(y_test, y_pred_test, model_name)

La funzione `build_model` costruisce e valuta un modello di regressione lineare. Prende in input i set di training e test, standardizza le feature con `scale_features`, addestra il modello specificato, genera le previsioni e chiama `evaluate_model` per stampare le metriche e visualizzare i risultati. Restituisce il modello addestrato.

In [ ]:
def build_model(X_train, X_test, y_train, y_test, model, model_name):
    """
    Build and evaluate a regression model.

    Args:
        X_train, X_test (array-like): Features.
        y_train, y_test (array-like): Target values.
        model: scikit-learn regression estimator.
        model_name (str): Label for output.

    Returns:
        Trained regression model.
    """
    X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

    model.fit(X_train_scaled, y_train)

    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)

    evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test)

    return model

La funzione `plot_learning_curve` visualizza la learning curve del modello, ovvero l'andamento dell'errore (MSE) su training e validation al variare della dimensione del training set. Prende in input un modello scikit-learn e i dati X, y.

La learning curve è uno strumento fondamentale per diagnosticare il **tradeoff bias-varianza**:
- Se le curve di training e validation sono entrambe alte con gap ridotto → il modello è in **underfitting** (alto bias)
- Se la curva di training è molto più bassa di quella di validation → il modello è in **overfitting** (alta varianza)
- Se le due curve convergono a un valore basso → il modello è ben calibrato

In [ ]:
def plot_learning_curve(model, X, y, title="", cv=5):
    """
    Plot learning curve to diagnose bias-variance tradeoff.

    Args:
        model: scikit-learn estimator (already includes scaling if needed).
        X (array-like): Features.
        y (array-like): Target values.
        title (str): Plot title.
        cv (int): Number of cross-validation folds.

    Returns:
        None
    """
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        cv=cv,
        scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )

    train_mse  = -train_scores.mean(axis=1)
    val_mse    = -val_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.figure(figsize=(9, 5), dpi=100)
    plt.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    plt.fill_between(train_sizes,
                     train_mse - train_std,
                     train_mse + train_std,
                     alpha=0.15, color='steelblue')
    plt.plot(train_sizes, val_mse, 'o-', color='tomato', label='Validation MSE')
    plt.fill_between(train_sizes,
                     val_mse - val_std,
                     val_mse + val_std,
                     alpha=0.15, color='tomato')
    plt.xlabel('Dimensione Training Set')
    plt.ylabel('MSE')
    plt.title(f'Learning Curve — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Caricamento del Dataset

Il dataset Diabetes di scikit-learn viene caricato e convertito in un DataFrame Pandas. Viene aggiunta la colonna `target` che rappresenta la misura quantitativa della progressione del diabete a un anno dalla raccolta dei dati clinici. Mostriamo le prime righe per una prima ispezione visiva.

In [ ]:
diabetes = load_diabetes()

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print(f"Dimensioni dataset: {df.shape}")
df.head()

## Statistiche descrittive

Calcoliamo le statistiche descrittive di base per comprendere la distribuzione di ciascuna variabile clinica: media, deviazione standard, valori minimi/massimi e percentili principali.

In [ ]:
df.describe()

## Controllo valori nulli

Verifichiamo l'assenza di valori nulli nel dataset. La presenza di dati mancanti richiederebbe strategie di imputation prima di procedere con l'addestramento del modello.

In [ ]:
df.isnull().sum()

# Analisi Esplorativa dei Dati (EDA)

## Distribuzione della variabile target

Analizziamo la distribuzione della variabile target tramite istogramma e boxplot. Una distribuzione approssimativamente normale è favorevole per i modelli di regressione lineare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].hist(df['target'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Progressione Diabete (target)')
axes[0].set_ylabel('Frequenza')
axes[0].set_title('Distribuzione del Target')

axes[1].boxplot(df['target'], vert=True)
axes[1].set_ylabel('Progressione Diabete (target)')
axes[1].set_title('Boxplot del Target')

plt.tight_layout()
plt.show()

print(f"Media: {df['target'].mean():.2f} | Mediana: {df['target'].median():.2f} | Std: {df['target'].std():.2f}")

## Scatter plot: Feature vs Target

Visualizziamo la relazione tra ciascuna feature clinica e il target. Le variabili con uno scatter più allungato presentano correlazione lineare più forte e saranno candidate privilegiate per il modello.

In [ ]:
feature_cols = diabetes.feature_names

fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df[col], df['target'], alpha=0.5, edgecolors='k', linewidths=0.2, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'{col} vs Target')

plt.suptitle('Relazione Feature-Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Matrice di correlazione — Dataset completo

La matrice di correlazione di Pearson mostra le relazioni lineari tra tutte le variabili. Particolare attenzione va posta all'ultima riga/colonna che mostra la correlazione di ciascuna feature con il target. Valori elevati di correlazione tra feature indicano possibile multicollinearità.

In [ ]:
print_correlation_matrix(df)

Estraiamo e ordiniamo le correlazioni con il target per identificare rapidamente le feature più predittive.

In [ ]:
corr_with_target = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(9, 5), dpi=100)
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_with_target.values]
plt.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Feature Clinica')
plt.ylabel('Correlazione con Target')
plt.title('Correlazione delle Feature con la Progressione del Diabete')
plt.tight_layout()
plt.show()

print(corr_with_target)

# Feature Engineering

## Analisi della multicollinearità

Dalla matrice di correlazione emergono alcune coppie di feature fortemente correlate tra loro:
- **s1 e s2** (colesterolo totale e LDL) mostrano correlazione positiva elevata
- **s3 e s4** (HDL e rapporto colesterolo/HDL) sono negativamente correlate
- **s1 e s3** sono correlate negativamente

La presenza di multicollinearità può destabilizzare i coefficienti della regressione lineare semplice. Costruiremo perciò due versioni del dataset:
- **Dataset completo** (`df`): tutte e 10 le feature
- **Dataset ridotto** (`df2`): esclude le feature più ridondanti, mantenendo quelle con correlazione più alta con il target e minore correlazione reciproca

Dal grafico di correlazione con il target, le feature meno predittive sono **s1**, **s2** e **sex**.

In particolare:
- **s1** e **s2** sono fortemente correlate tra loro (multicollinearità) e meno correlate al target rispetto a **s5** e **bmi**
- **sex** mostra la correlazione più debole con il target

Creiamo `df2` rimuovendo queste colonne per ottenere un dataset più pulito da un punto di vista statistico.

In [ ]:
df2 = df.drop(['s1', 's2', 'sex'], axis=1)
print(f"Feature dataset completo:  {list(df.drop('target', axis=1).columns)}")
print(f"Feature dataset ridotto:   {list(df2.drop('target', axis=1).columns)}")

## Matrice di correlazione — Dataset ridotto

Verifichiamo che il dataset ridotto presenti minore multicollinearità tra le feature rimaste.

In [ ]:
print_correlation_matrix(df2)

## Dataset per l'identificazione del modello

Creiamo le variabili X e y per entrambe le versioni del dataset. `X` contiene le feature e `y` il target. I due dataset verranno usati per confrontare le prestazioni dei modelli con e senza le feature ridondanti.

In [ ]:
X   = df.drop('target', axis=1).values
y   = df['target'].values

X_2 = df2.drop('target', axis=1).values
y_2 = df2['target'].values

print(f"Shape dataset completo:  X={X.shape}, y={y.shape}")
print(f"Shape dataset ridotto:   X={X_2.shape}, y={y_2.shape}")

# Regression Models

Testiamo quattro varianti della regressione lineare, ciascuna con una diversa strategia di regolarizzazione:

| Modello | Regolarizzazione | Note |
|---------|-----------------|------|
| Linear Regression | Nessuna (baseline) | Massima varianza, rischio overfitting |
| Ridge | L2 | Penalizza i coefficienti grandi, gestisce la multicollinearità |
| Lasso | L1 | Porta alcuni coefficienti a zero (selezione feature automatica) |
| ElasticNet | L1 + L2 | Combinazione bilanciata dei vantaggi di Ridge e Lasso |

Per ciascun modello confrontiamo le prestazioni sul **dataset completo** e sul **dataset ridotto**.

La suddivisione train/test è 80%/20% con `random_state=0` per garantire la riproducibilità.

## Linear Regression (baseline)

La regressione lineare semplice senza regolarizzazione costituisce il punto di riferimento (baseline) per tutti i modelli successivi. Non penalizza i coefficienti, quindi è la più sensibile alla multicollinearità e al rumore.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, LinearRegression(), "Linear Regression — completo")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, LinearRegression(), "Linear Regression — ridotto")

### Conclusioni Linear Regression
La regressione lineare senza regolarizzazione mostra già un R² positivo, confermando che le feature cliniche hanno potere predittivo sulla progressione del diabete. La differenza tra dataset completo e ridotto indica se la multicollinearità stava penalizzando il modello base.

## Ridge Regression (L2)

La regressione Ridge aggiunge una penalità L2 alla funzione di costo, proporzionale alla somma dei quadrati dei coefficienti. Questo riduce la varianza del modello e gestisce la multicollinearità distribuendo il peso tra feature correlate, senza azzerare completamente nessun coefficiente.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, Ridge(alpha=1.0), "Ridge (alpha=1.0) — completo")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, Ridge(alpha=1.0), "Ridge (alpha=1.0) — ridotto")

### Conclusioni Ridge
La regolarizzazione L2 dovrebbe ridurre l'overfitting rispetto alla regressione lineare semplice, con un miglioramento più marcato sul dataset completo dove la multicollinearità era più presente. Sul dataset ridotto la differenza con la baseline tende a essere meno netta, poiché i dati erano già meno collineari.

## Lasso Regression (L1)

La regressione Lasso aggiunge una penalità L1 proporzionale alla somma dei valori assoluti dei coefficienti. A differenza di Ridge, Lasso porta alcuni coefficienti esattamente a zero, effettuando una selezione automatica delle feature più rilevanti. Questo la rende particolarmente utile quando si sospetta che non tutte le feature siano informative.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

lasso_full = build_model(X_train, X_test, y_train, y_test, Lasso(alpha=0.1, max_iter=10000), "Lasso (alpha=0.1) — completo")

print("Coefficienti Lasso (dataset completo):")
for name, coef in zip(diabetes.feature_names, lasso_full.coef_):
    print(f"  {name:>5}: {coef:.4f}{'  ← azzerato' if coef == 0 else ''}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

lasso_reduced = build_model(X_train, X_test, y_train, y_test, Lasso(alpha=0.1, max_iter=10000), "Lasso (alpha=0.1) — ridotto")

print("Coefficienti Lasso (dataset ridotto):")
for name, coef in zip(df2.drop('target', axis=1).columns, lasso_reduced.coef_):
    print(f"  {name:>5}: {coef:.4f}{'  ← azzerato' if coef == 0 else ''}")

### Conclusioni Lasso
I coefficienti azzerati da Lasso confermano quali feature il modello considera non informative per la previsione della progressione del diabete. Questo è coerente con l'analisi della correlazione con il target eseguita nella fase EDA. Lasso offre il vantaggio aggiuntivo di produrre un modello più interpretabile e parsimonioso.

## ElasticNet (L1 + L2)

ElasticNet combina le penalità L1 (Lasso) e L2 (Ridge) tramite il parametro `l1_ratio`. Con `l1_ratio=0.5` viene bilanciato equamente il contributo delle due regolarizzazioni: si ottiene sia la selezione delle feature tipica di Lasso, sia la stabilità sui gruppi di feature correlate tipica di Ridge.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000), "ElasticNet (alpha=0.1, l1=0.5) — completo")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

build_model(X_train, X_test, y_train, y_test, ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000), "ElasticNet (alpha=0.1, l1=0.5) — ridotto")

### Conclusioni ElasticNet
ElasticNet si comporta in modo intermedio tra Ridge e Lasso. Sul dataset completo, la componente L2 aiuta a gestire la multicollinearità mentre la componente L1 azzera le feature meno rilevanti. Sul dataset ridotto il vantaggio rispetto a Ridge tende a ridursi poiché la multicollinearità era già stata affrontata manualmente.

# Conclusioni Finali e Scelta del Modello

## Dataset finale

Il dataset finale scelto per il modello di produzione è il **dataset ridotto** (`df2`), che esclude le feature con multicollinearità elevata (s1, s2) e bassa correlazione con il target (sex). Questa scelta garantisce un modello più stabile e interpretabile.

In [ ]:
dataset_final = df2.copy()
print(f"Feature del modello finale: {list(dataset_final.drop('target', axis=1).columns)}")
print(f"Shape: {dataset_final.shape}")
dataset_final.head()

## Modello finale

Il modello selezionato è la **Ridge Regression** sul dataset ridotto, che ha mostrato il miglior bilanciamento tra bias e varianza nei test precedenti.

In [ ]:
X_final = dataset_final.drop('target', axis=1).values
y_final = dataset_final['target'].values

print(f"Shape X: {X_final.shape}")
print(f"Shape y: {y_final.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=0)
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

final_model = Ridge(alpha=1.0)
final_model.fit(X_train_scaled, y_train)

y_pred_train = final_model.predict(X_train_scaled)
y_pred_test  = final_model.predict(X_test_scaled)

evaluate_model("Ridge — Modello Finale", y_train, y_test, y_pred_train, y_pred_test)

## Cross-Validation

Eseguiamo una **k-fold cross-validation** (k=5) sull'intero dataset ridotto. Rispetto alla singola suddivisione hold-out, la cross-validation fornisce una stima più robusta della generalizzazione perché ogni osservazione viene usata sia per il training che per la validation, riducendo la dipendenza dalla casualità della singola divisione.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

cv_scores = cross_val_score(pipeline, X_final, y_final, cv=5, scoring='r2')

print(f"R² per fold: {np.round(cv_scores, 4)}")
print(f"R² medio:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

plt.figure(figsize=(8, 4), dpi=100)
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Media R²={cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R²')
plt.title('K-Fold Cross-Validation R² — Ridge Modello Finale')
plt.legend()
plt.tight_layout()
plt.show()

## Learning Curve — Analisi del Tradeoff Bias-Varianza

La learning curve mostra come variano l'errore di training e di validation al crescere della dimensione del training set. È lo strumento principale per diagnosticare il tradeoff bias-varianza:

- **Alto bias (underfitting):** entrambe le curve convergono a un valore di errore alto — il modello non ha sufficiente capacità per catturare il pattern nei dati
- **Alta varianza (overfitting):** la curva di training è molto più bassa di quella di validation — il modello si adatta troppo ai dati di training e generalizza male
- **Modello ben calibrato:** le due curve convergono a un valore di errore basso con gap contenuto

In [ ]:
plot_learning_curve(pipeline, X_final, y_final, title="Ridge — Dataset Ridotto", cv=5)

Confrontiamo la learning curve di Ridge con quella della regressione lineare semplice (baseline) per visualizzare l'effetto della regolarizzazione L2 sul tradeoff bias-varianza.

In [ ]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)

for ax, pipe, title in [
    (axes[0], pipeline_lr, "Linear Regression (baseline)"),
    (axes[1], pipeline,    "Ridge (L2 — alpha=1.0)")
]:
    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_final, y_final,
        cv=5, scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mse = -train_scores.mean(axis=1)
    val_mse   = -val_scores.mean(axis=1)

    ax.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    ax.plot(train_sizes, val_mse,   'o-', color='tomato',    label='Validation MSE')
    ax.set_xlabel('Dimensione Training Set')
    ax.set_ylabel('MSE')
    ax.set_title(f'Learning Curve — {title}')
    ax.legend()

plt.tight_layout()
plt.show()

## Esportazione del Modello

Il modello finale viene esportato tramite `pickle` insieme allo scaler e ai metadati necessari per la produzione. Questo permette di ricaricare il modello nella piattaforma sanitaria di MedPredict e applicarlo su nuovi dati clinici, garantendo che lo stesso pre-processing venga applicato in fase di inferenza.

In [ ]:
scaler_final = StandardScaler()
scaler_final.fit(X_train)

model_artifact = {
    'model': final_model,
    'scaler': scaler_final,
    'features': list(dataset_final.drop('target', axis=1).columns),
    'model_name': 'Ridge (alpha=1.0)',
    'test_r2': float(r2_score(y_test, y_pred_test)),
    'test_rmse': float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
}

with open('medpredict_diabetes_model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Modello salvato in: medpredict_diabetes_model.pkl")
print(f"Modello: {model_artifact['model_name']}")
print(f"Feature: {model_artifact['features']}")
print(f"R² Test: {model_artifact['test_r2']:.4f}")
print(f"RMSE Test: {model_artifact['test_rmse']:.2f}")

In [ ]:
with open('medpredict_diabetes_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

y_pred_check = loaded['model'].predict(loaded['scaler'].transform(X_test))
print(f"Previsioni identiche dopo il ricaricamento: {np.allclose(y_pred_test, y_pred_check)}")
print("Modello pronto per la piattaforma sanitaria MedPredict.")

## Conclusioni Finali

Riassumendo il processo seguito:

- Il dataset Diabetes di scikit-learn è stato esplorato tramite statistiche descrittive, scatter plot e matrice di correlazione, identificando le feature più predittive (**bmi**, **s5**, **bp**) e le coppie con multicollinearità elevata (**s1/s2**, **s3/s4**).

- Sono stati costruiti e confrontati quattro modelli di regressione lineare con diversa regolarizzazione (**Linear Regression, Ridge, Lasso, ElasticNet**), ciascuno testato sia sul dataset completo che su quello ridotto.

- La **learning curve** ha confermato che il modello Ridge sul dataset ridotto presenta il miglior bilanciamento bias-varianza: gap ridotto tra training e validation MSE, con entrambe le curve che convergono stabilmente all'aumentare dei dati disponibili.

- La **k-fold cross-validation** (k=5) ha prodotto un R² medio stabile, confermando che le prestazioni non dipendono dalla singola suddivisione hold-out.

- Il modello è stato esportato con **pickle** ed è pronto per essere integrato nella piattaforma sanitaria di MedPredict.

### Alcune cifre di massima
- R² medio in cross-validation: ~0.49–0.53
- Il modello spiega circa il 50% della varianza della progressione del diabete con sole variabili cliniche di base
- L'aggiunta di dati longitudinali (storico del paziente) e di biomarcatori aggiuntivi potrebbe incrementare significativamente la capacità predittiva del modello in produzione